# 05 — Geometric observables

Northwest splitting pairs N–W and S–E dual edges at alternating plaquettes. The NE–SW spin diagonal is connected for either spin value. Ordinary nearest-neighbor labeling is shown only for comparison; it is not the production default. Physical interactions and FK bonds remain nearest-neighbor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.ndimage import label, binary_fill_holes
from ising.observables import (NW_CONNECTIVITY, plus_components, droplet_mask,
    peierls_contours, droplet_geometry, interface_length, interface_heights,
    boundary_magnetization, cluster_statistics)
from ising import IsingModel, Simulator

spin_cmap = ListedColormap(['#2c4a6b', '#e3952f'])
NW_CONNECTIVITY.astype(int)

## Alternating plaquettes
Compare the two diagonal orientations. Label zero is background; positive labels identify components.

In [ ]:
for interior in (np.array([[-1, 1], [1, -1]]), np.array([[1, -1], [-1, 1]])):
    nearest_neighbor_labels, _ = label(interior == 1)
    northwest_labels, _ = label(interior == 1, structure=NW_CONNECTIVITY)
    print(interior, nearest_neighbor_labels, northwest_labels, sep='\n')

## Diagonal escape channels
A chain of minus sites reaches the exterior through the upper-right corner (NE–SW) or upper-left corner (NW–SE). Only the former remains open under the chosen convention.

In [ ]:
escape_ne_sw = np.ones((9, 9), dtype=np.int8)
escape_nw_se = escape_ne_sw.copy()
for i in range(5):
    escape_ne_sw[i, 8-i] = -1
    escape_nw_se[i, i] = -1
state = np.pad(escape_ne_sw, 1, constant_values=-1)
plus_mask = state[1:-1, 1:-1] == 1

## Labeling and size extraction
SciPy labels the plus sites with the shared six-neighbor structure. `bincount` counts each label; discard the background bin.

In [ ]:
component_labels, num_components = label(plus_mask, structure=NW_CONNECTIVITY)
component_sizes = np.bincount(component_labels.ravel())[1:]
production_labels, production_sizes = plus_components(state)
assert np.array_equal(component_labels, production_labels)
assert np.array_equal(component_sizes, production_sizes)
assert component_sizes.sum() == plus_mask.sum()
component_sizes

## Largest occupied component and filling
Select by occupied site count, not enclosed area. `argmax` selects the first label on ties. The same structure determines which background sites can escape.

In [ ]:
largest_label = int(np.argmax(component_sizes)) + 1
largest_component = component_labels == largest_label
filled_droplet = binary_fill_holes(largest_component, structure=NW_CONNECTIVITY)
assert np.array_equal(filled_droplet, droplet_mask(component_labels, component_sizes))
print(cluster_statistics(component_labels, component_sizes))
print(droplet_geometry(filled_droplet))

## Explicit contours and mask agreement
Each ordered walk is closed, with plus on the right. Half-integer coordinates refer to the padded array. Raw plots may touch at alternating plaquettes even though the ordered edges resolve them.

In [ ]:
contours = peierls_contours(state)
assert sum(len(contour)-1 for contour in contours) == interface_length(state)
filled_state = np.pad(np.where(filled_droplet, 1, -1), 1, constant_values=-1)
outer_contours = peierls_contours(filled_state)
assert len(outer_contours) == 1

def directed_edges(contour):
    return {(tuple(a), tuple(b)) for a, b in zip(contour[:-1], contour[1:])}

assert any(directed_edges(outer_contours[0]) == directed_edges(c) for c in contours)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, interior in enumerate((escape_ne_sw, escape_nw_se)):
    example_state = np.pad(interior, 1, constant_values=-1)
    labels, sizes = plus_components(example_state)
    largest = labels == np.argmax(sizes)+1
    filled = droplet_mask(labels, sizes)
    for ax, data, title in zip(axes[row], (interior, labels, largest, filled, interior),
                              ('Spins', 'Labels', 'Largest component', 'Filled droplet', 'Contours')):
        ax.imshow(data, cmap=spin_cmap if title in ('Spins', 'Contours') else 'viridis',
                  extent=(.5, 9.5, 9.5, .5), interpolation='nearest')
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
    for contour in peierls_contours(example_state):
        axes[row, 4].plot(contour[:, 1], contour[:, 0], color='white', linewidth=1.5)
    assert filled.sum() == (76 if row == 0 else 80)
plt.tight_layout()
plt.show()

## Simulation observables
The simulator reuses one labeling and one mask. Boundary magnetization is independent of geometry; first-hit heights depend on the filled mask. Missing rays are `None`. Radius of gyration is the root mean squared distance from the mask centroid.

In [ ]:
sim = Simulator(IsingModel(32, .5, 3), init='plus', seed=42)
sim.advance(10)
measurements = sim.observe(cluster_details=True, contours=True, boundary_profile=True,
                           interface_heights=True)
measurements